# 🌿 AgriVision Pro — Colab Backend Server

This notebook runs the **FastAPI backend** of AgriVision Pro on Google Colab's free GPU, then exposes it via **ngrok** so your local React frontend can connect to it.

## Architecture Overview

```
┌──────────────────────────────┐         ┌──────────────────────────────────┐
│   YOUR LOCAL MACHINE         │         │   GOOGLE COLAB (Free GPU)        │
│                              │         │                                  │
│   React Frontend (Vite)      │ ──────> │   FastAPI Backend (main.py)      │
│   localhost:5173             │  HTTPS  │   uvicorn :8000                  │
│                              │  ngrok  │   + TensorFlow models            │
│   .env.local:               │  tunnel │   + scikit-learn models          │
│   VITE_API_BASE_URL=<ngrok> │         │   + OpenCV processing            │
└──────────────────────────────┘         └──────────────────────────────────┘
```

## Steps
1. **Clone the repo** and install dependencies
2. **Train the models** (ML + DL pipeline)
3. **Install ngrok** and authenticate
4. **Start the FastAPI server** with ngrok tunnel
5. **Copy the ngrok URL** into your local frontend `.env.local`
6. **Run the frontend locally** with `npm run dev`

---
## Step 1 — Enable GPU Runtime

> **Before running any cell**, go to:
>
> `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`
>
> This gives you a free GPU for TensorFlow inference and training.

In [ ]:
# Verify GPU is available
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detected: {gpus[0].name}")
    print(f"   Device: {tf.test.gpu_device_name()}")
else:
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → GPU")
    print("   The pipeline will still work on CPU, but training will be slower.")

---
## Step 2 — Clone the Repository

In [ ]:
import os

REPO_URL = "https://github.com/ouemnaa/Plant_Disease_Project.git"  # ← Change if your repo URL is different
REPO_DIR = "/content/Plant_Disease_Project"

if os.path.exists(REPO_DIR):
    print(f"📂 Repository already cloned at {REPO_DIR}")
    print("   Pulling latest changes...")
    !cd {REPO_DIR} && git pull
else:
    print(f"📥 Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Working directory: {os.getcwd()}")
!ls -la

---
## Step 3 — Install Python Dependencies

Colab already has TensorFlow, NumPy, and scikit-learn pre-installed.  
We only need to install the missing packages: `fastapi`, `uvicorn`, `python-multipart`, `pyngrok`, etc.

In [ ]:
# Install project dependencies (skips already-installed packages)
!pip install -q fastapi uvicorn[standard] python-multipart pyngrok joblib scikit-image opencv-python-headless

# Verify key imports
import fastapi, uvicorn, cv2, joblib
print(f"✅ FastAPI: {fastapi.__version__}")
print(f"✅ OpenCV:  {cv2.__version__}")
print(f"✅ All dependencies installed successfully")

---
## Step 4 — Train the Models

This step runs the full training pipeline:
- **Classical ML** (Random Forest + SVM) — extracts color/texture features from up to 20k images
- **Deep Learning** (MobileNetV2) — transfer learning on 38 PlantVillage classes
- **Tomato Specialist** — fine-tuned model for 10 tomato diseases

> ⏱️ **Expected time**: ~10–15 min on GPU (first run only).  
> If models already exist in `outputs/`, this step is **skippable**.

In [ ]:
from pathlib import Path

# Check if models already exist
ml_exists = Path("outputs/notebook_run/best_ml_model.joblib").exists()
dl_exists = Path("outputs/notebook_run/dl/dl_model.keras").exists()
tomato_exists = Path("outputs/tomato_model/tomato_model.keras").exists()

print("📊 Model Status:")
print(f"   ML Model (RF/SVM):    {'✅ Found' if ml_exists else '❌ Not found — will train'}")
print(f"   Full DL (38 classes): {'✅ Found' if dl_exists else '❌ Not found — will train'}")
print(f"   Tomato DL (10 cls):   {'✅ Found' if tomato_exists else '❌ Not found — will train'}")

In [ ]:
# ── Train Full Pipeline (ML + DL) ──
# Skip if both models already exist

if not ml_exists or not dl_exists:
    print("🚀 Running full training pipeline (ML + DL)...")
    print("   This will download PlantVillage from TFDS and train all models.")
    print("   ⏱️  Estimated time: 10-15 minutes on GPU\n")
    !python run_pipeline.py --run-dl --max-samples 20000 --dl-epochs 5 --output outputs/notebook_run
else:
    print("✅ Full pipeline models already exist. Skipping training.")
    print("   Delete outputs/notebook_run/ to retrain.")

In [ ]:
# ── Train Tomato Specialist ──

if not tomato_exists:
    print("🍅 Training Tomato Specialist model...")
    print("   ⏱️  Estimated time: 3-5 minutes on GPU\n")
    !python train_tomato_model.py --epochs 5
else:
    print("✅ Tomato model already exists. Skipping training.")
    print("   Delete outputs/tomato_model/ to retrain.")

In [ ]:
# ── Verify all models are ready ──

print("\n" + "="*50)
print("📋 FINAL MODEL CHECK")
print("="*50)

models_ok = True
for label, path in [
    ("ML Model (RF/SVM)",        "outputs/notebook_run/best_ml_model.joblib"),
    ("Full DL (38 classes)",     "outputs/notebook_run/dl/dl_model.keras"),
    ("Tomato DL (10 classes)",   "outputs/tomato_model/tomato_model.keras"),
    ("Run Summary (metadata)",   "outputs/notebook_run/run_summary.json"),
    ("Tomato Meta",              "outputs/tomato_model/tomato_model_meta.json"),
]:
    exists = Path(path).exists()
    status = "✅" if exists else "❌"
    print(f"   {status} {label}: {path}")
    if not exists:
        models_ok = False

if models_ok:
    print("\n🎉 All models ready! Proceed to start the API server.")
else:
    print("\n⚠️ Some models are missing. Re-run the training cells above.")

---
## Step 5 — Setup ngrok Tunnel

**ngrok** creates a public HTTPS URL that forwards traffic to the Colab backend.  
Your local frontend will use this URL as `VITE_API_BASE_URL`.

### 🔑 Get your free ngrok auth token:
1. Go to [https://dashboard.ngrok.com/signup](https://dashboard.ngrok.com/signup) — sign up (free)
2. Go to [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Copy your auth token and paste it below

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  PASTE YOUR NGROK AUTH TOKEN BELOW                           ║
# ╚════════════════════════════════════════════════════════════════╝

NGROK_AUTH_TOKEN = ""  # ← Paste your token here (e.g. "2abc123def456...")

# ─────────────────────────────────────────────────────────────────

if not NGROK_AUTH_TOKEN:
    print("❌ Please set your NGROK_AUTH_TOKEN above!")
    print("   Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print("✅ ngrok authenticated successfully")

---
## Step 6 — Start the FastAPI Server + ngrok Tunnel 🚀

This cell:
1. Opens an ngrok tunnel on port `8000`
2. Starts the FastAPI server (`main.py`) with uvicorn
3. Prints the **public URL** you need for your frontend

> ⚠️ **This cell runs indefinitely** (it's a server). To stop it, use `Runtime → Interrupt execution`.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn

# Kill any existing ngrok tunnels
ngrok.kill()

# Open a new tunnel
public_url = ngrok.connect(8000, "http")
public_url_str = str(public_url).replace('"', '').replace("NgrokTunnel: ", "").split(" ->")[0].strip()

# If pyngrok returns an object, extract the public_url attribute
if hasattr(public_url, 'public_url'):
    public_url_str = public_url.public_url

print("\n" + "═"*60)
print("🌿 AgriVision Pro — Backend Server")
print("═"*60)
print(f"\n🔗 PUBLIC URL:  {public_url_str}")
print(f"📡 API Docs:    {public_url_str}/docs")
print(f"🔬 Health:      {public_url_str}/api/info")
print("\n" + "─"*60)
print("📋 NEXT STEPS (on your local machine):")
print("─"*60)
print(f"\n1. Open frontend/.env.local and set:")
print(f"   VITE_API_BASE_URL={public_url_str}")
print(f"\n2. Start the frontend:")
print(f"   cd frontend")
print(f"   npm run dev")
print(f"\n3. Open http://localhost:5173 in your browser")
print("\n" + "─"*60)
print("⚠️  Keep this cell running! The server stops when you stop this cell.")
print("═"*60 + "\n")

# Start the FastAPI server (blocks until interrupted)
uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="info")

---
## 🛑 Cleanup (Optional)

Run this after you're done to close the ngrok tunnel.

In [ ]:
from pyngrok import ngrok
ngrok.kill()
print("✅ ngrok tunnel closed.")

---
## 📖 Troubleshooting

### Common Issues

| Problem | Solution |
|---------|----------|
| **Frontend can't connect** | Make sure you copied the ngrok URL into `frontend/.env.local` and restarted `npm run dev` |
| **ngrok error: too many tunnels** | Run the Cleanup cell above, then re-run Step 6 |
| **CORS errors in browser** | The backend already allows all origins (`*`). Try hard-refreshing (`Ctrl+Shift+R`) |
| **Models not found** | Re-run Step 4 training cells |
| **Colab disconnected** | Colab has idle timeouts (~90 min). Re-run from Step 6 (models persist in the runtime) |
| **ngrok URL changed** | ngrok gives a new URL each time. Update `frontend/.env.local` and restart the dev server |
| **ERR_NGROK_6024 (blocked by browser)** | Click the ngrok interstitial page "Visit Site" button once, or add the header `ngrok-skip-browser-warning: true` |

### API Endpoints

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET` | `/api/info` | Returns loaded models status and class names |
| `POST` | `/api/analyze` | Upload an image → returns ML/DL predictions + processing steps |
| `GET` | `/docs` | Interactive Swagger API documentation |

### Testing the API

You can test the API directly from your browser by visiting `<ngrok-url>/docs` and using the interactive Swagger UI.